# BlindSpot v0.3.1: train, audit, reproduce

CPU-only candidate suite with trajectory-derived movement, explicit speed-unit consistency checks, factorized possession inference, symmetric gradient boosting, whole-run purged cross-fitting and uncertainty-driven review. No GPU purchase or hosted LLM is required. Public data is a previously inspected development example; these experiments do not establish a private-match ranking.

Audit revision: observation masks distinguish missing measurements from zero; team inference resets at algorithmically inferred stoppages. Owner support is conditional on the decoded live/dead segmentation. Additional fixed-time comparisons are a same-sample sensitivity audit, not independent validation.


In [ ]:
%pip -q install numpy==2.2.6 pandas==2.2.3 scipy==1.17.1 scikit-learn==1.8.0 joblib==1.5.3 threadpoolctl==3.6.0 gdown==5.2.0


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile, subprocess, sys
import gdown

PROJECT = Path('/content/blindspot-v031')
PROJECT.mkdir(exist_ok=True)
BUNDLE_ID = '19N61uNYD06YHDL12w-sdOVVAJgGc2Jx-'
BUNDLE_SHA256 = 'eba9fefb39422fa66b2a6e209ccdf6ebfc2982f9153a73b4c449a1a998ef49d0'
def sha256(path):
    value = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()
package = Path('/content/BlindSpot_v031_project.zip')
if not package.exists() or sha256(package) != BUNDLE_SHA256:
    result = gdown.download(id=BUNDLE_ID, output=str(package), quiet=False)
    if result is None:
        raise RuntimeError('Bundle download failed; check sharing or Drive bandwidth limits')
assert sha256(package) == BUNDLE_SHA256, 'Project bundle checksum mismatch'
with zipfile.ZipFile(package) as archive:
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe bundle path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
print('Verified source bundle:', PROJECT)
os.environ['LOKY_MAX_CPU_COUNT'] = '2'


In [ ]:
ASSETS = {'BAR-ATH_tactical_atd.txt': '1JEPmnLYax4dnVh1ZlInJAH5calmS1tIz', 'BAR-ATH_tactical_atd.xml': '1WBfy2sOwmJ2cX2w9XJq6y4VXo_kxKHjX', 'BAR-ATH_tactical_homography.csv': '1tuq2EGFmfvnXl8u5dU7JLoN3eNbrWwDJ', 'BAR-ATH_tactical_smart_tagging.xml': '1SvfntIRtFh6i7KAK1gVJk1NQbS_lus2K'}
EXPECTED = {'BAR-ATH_tactical_atd.txt': 'f130e350d2bfdf734b7e32f072931fccc6d015c1e837d082064875fe0a700431', 'BAR-ATH_tactical_atd.xml': '327e35e32e46a647389b88f883d4f9a2ab931e3e6148aef2fcf8f22b1e93c43a', 'BAR-ATH_tactical_homography.csv': '421c81abe16f0912dc45c2541f9bf19dd5660066397a3dcc043038a1b61c28c3', 'BAR-ATH_tactical_smart_tagging.xml': '0594d35e3df01b9c5002372d8615e2e8ef5379ac8f4e84c5ef7a53f4304c9019'}
data_dir = PROJECT / 'data/tactical'
data_dir.mkdir(parents=True, exist_ok=True)
for name, file_id in ASSETS.items():
    path = data_dir / name
    if not path.exists() or sha256(path) != EXPECTED[name]:
        result = gdown.download(id=file_id, output=str(path), quiet=False)
        if result is None:
            raise RuntimeError('Download failed: ' + name)
    assert sha256(path) == EXPECTED[name], 'Dataset checksum mismatch: ' + name
    print('Verified', name, path.stat().st_size, 'bytes')


## Optional durable output
Default output is temporary runtime storage. Set PERSIST_TO_DRIVE=True and authorize your own Drive mount to retain verified stages across runtime resets. This does not change sharing. Otherwise download the final ZIP.


In [ ]:
PERSIST_TO_DRIVE = False
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/BlindSpot/checkpoints/v031')
else:
    OUT = PROJECT / 'data/experiments/blindspot-v031'
OUT.mkdir(parents=True, exist_ok=True)
print('Output:', OUT, '| durable:', PERSIST_TO_DRIVE)


## Correctness checks

Tests cover exact inference, unit semantics, trajectory continuity, restart ownership, team-swap invariance, training-label isolation, complete-run fold purging, halftime reset and corrupted/interrupted checkpoint recovery.


In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## Train and evaluate the fixed candidates

Three fixed approaches, no large hyperparameter search. Four contiguous evaluation folds preserve entire H/A/D runs; training excludes those folds and neighbouring runs with a 300-frame embargo. Report pooled matching counts over the same folds for all methods. The final exported classifier is fitted on ALL public labels; its training-window score is descriptive, not validation. Older metadata-scaling pilot results are not the trajectory-based result.


In [ ]:
command = [sys.executable, '-u', 'scripts/run_champion.py', '--config', 'configs/champion_v031.json', '--out', str(OUT), '--resume', '--archive']
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
if status:
    raise subprocess.CalledProcessError(status, command)


In [ ]:
subprocess.run([sys.executable, 'evaluation/validate_submission.py', str(OUT / 'candidate/CoreOrigin.csv')], check=True)
results = json.loads((OUT / 'supervised/pooled_scores.json').read_text())
import pandas as pd
display(pd.DataFrame(results).T[['macro_f1','spell_f1_mean','dead_f1','n_graded','scope']])
print('SPEED UNIT AUDIT')
print((OUT / 'evaluation/speed_unit_audit.json').read_text())
print('OUTPUT_CSV_HASHES')
print(json.dumps(json.loads((OUT / 'manifest.json').read_text())['output_csv_sha256'], indent=2))


## Inspect the result

Colab shows a static dashboard preview. Download and extract the archive for working local links and the interactive evidence report. The default cloud run does not download videos or build clips. Local video demo: install requirements-v3-video.txt and run scripts/build_candidate_review.py with permitted tactical assets.


In [ ]:
from IPython.display import HTML, display
import re
preview = (OUT / 'index.html').read_text()
preview = re.sub(r'href="[^"]*"', '', preview)
display(HTML('<p>Static Colab preview. Download the result archive for working local links.</p>' + preview))


## Save results before disconnecting

The archive includes the provisional candidate CSV, source/input/output hashes, fitted models, four-fold development evidence, unit audit, missing-ball stress test and review risk. Model files were generated by this notebook; only load trusted model artifacts. Keep match-derived results within the permitted sharing scope.


In [ ]:
import shutil
from google.colab import files
result_zip = shutil.make_archive('/content/BlindSpot_v031_results', 'zip', OUT)
files.download(result_zip)
